In [ ]:

# -- Cell 1 -- rclone + Drive.
# Settings -> Internet ON, Accelerator GPU T4 x2, RCLONE_DRIVE_TOKEN attached.
import os, subprocess

r = subprocess.run("curl -s https://rclone.org/install.sh | sudo bash", shell=True)
if r.returncode not in (0, 3):
    raise RuntimeError("rclone install failed (exit %d)" % r.returncode)

from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("RCLONE_DRIVE_TOKEN")
os.makedirs("/root/.config/rclone", exist_ok=True)
with open("/root/.config/rclone/rclone.conf", "w") as f:
    f.write("[drive]\ntype = drive\nscope = drive\ntoken = " + token + "\n")

REMOTE = "drive:Distillation"
out = subprocess.run("rclone lsf " + REMOTE, shell=True, capture_output=True, text=True)
print(out.stdout or out.stderr)
assert out.returncode == 0, "cannot see " + REMOTE


In [ ]:

# -- Cell 2 -- DOES BI-GUIDED SELECTION BEAT NAIVE TRUNCATION AT THE SAME SIZE?
#
# Two 8-block models, both 84.8M parameters, differing only in WHICH 8:
#
#   trunc8   blocks 0-7                    the first 8, no selection
#   bisel8   blocks 0,1,2,3,5,6,10,16      block 0 plus the 7 highest-BI blocks
#                                          from 1-16
#
# Capacity is held constant, so the comparison isolates the value of the Block
# Influence ranking itself. If bisel8 wins, BI is telling us something; if they
# tie, BI is an expensive way to reproduce "keep the first k".
#
# SEAM SAFETY, from the 1200-molecule residual-norm profile. Skipping blocks means
# a kept block receives a stream built by fewer predecessors than it trained on.
# Tracking the compounding norms for bisel8: block 5 receives ~177 where it expects
# ~192 (1.08x), block 10 receives ~202 vs 244 (1.21x), block 16 receives ~211 vs
# 293 (1.39x). All far below the level that broke suffix16 (39.6x, which scored
# 0.5593 -- below a bag-of-tokens baseline). So any difference here is about block
# choice, not about a distribution cliff.
subprocess.run('pip install -q -U "transformers>=5.0" peft lightning', shell=True, check=True)
subprocess.run("pip uninstall -y -q torchao", shell=True)

import torch, numpy as np, pandas as pd, glob, json, time
print("torch", torch.__version__, "| GPUs", torch.cuda.device_count())
NGPU = max(1, torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print("   cuda:%d %s %.0f GB" % (i, p.name, p.total_memory / 1e9))


In [ ]:

# -- Cell 3 -- code, data, teacher.
WORK = "/kaggle/working"
CODE, REPO = WORK + "/distill", WORK + "/project/their_repo"
TEACH = WORK + "/models/peptideclm-2-mlm-large"
SMALL = WORK + "/models/peptideclm-2-mlm-small"
os.makedirs(REPO, exist_ok=True)

def rlsf(path):
    r = subprocess.run("rclone lsf " + path, shell=True, capture_output=True, text=True)
    return r.stdout.split() if r.returncode == 0 else []

def pull_model(name, local):
    if os.path.exists(local + "/model.safetensors"):
        return "already local"
    flat = "%s/models/%s" % (REMOTE, name)
    if any(f.startswith("model.safetensors") for f in rlsf(flat)):
        os.makedirs(local, exist_ok=True)
        subprocess.run("rclone copy %s %s -P" % (flat, local), shell=True, check=True)
        return "flat"
    snaps = "%s/models/models--aaronfeller--%s/snapshots" % (REMOTE, name)
    shas = [x.rstrip("/") for x in rlsf(snaps)]
    assert shas, "%s not found. Tried  %s  and  %s" % (name, flat, snaps)
    os.makedirs(local, exist_ok=True)
    subprocess.run("rclone copy %s/%s %s -P" % (snaps, shas[0], local), shell=True, check=True)
    return "snapshot " + shas[0][:12]

for sub in ("data", "training"):
    if not os.path.isdir(REPO + "/" + sub):
        subprocess.run("rclone copy %s/their_repo/%s %s/%s --transfers 16 -P"
                       % (REMOTE, sub, REPO, sub), shell=True, check=True)
subprocess.run("rclone copy %s/distill %s --transfers 8 -P" % (REMOTE, CODE),
               shell=True, check=True)
print("teacher <- %s" % pull_model("peptideclm-2-mlm-large", TEACH))
print("small   <- %s" % pull_model("peptideclm-2-mlm-small", SMALL))

TRAIN_PY = REPO + "/training/02_classification_benchmarks_training_code/scripts/classification_finetuning_v2.py"
DATA_DIR = REPO + "/data"
if not os.path.exists(WORK + "/their_repo"):
    os.symlink(REPO, WORK + "/their_repo")
for name, src in (("peptideclm-2-mlm-large", TEACH), ("peptideclm-2-mlm-small", SMALL)):
    d = "%s/models/models--aaronfeller--%s/snapshots/local" % (WORK, name)
    if not os.path.exists(d):
        os.makedirs(os.path.dirname(d), exist_ok=True)
        os.symlink(src, d)

# writes THPep_train.csv / THPep_test.csv, which their script needs
r = subprocess.run(["python", "bench_control.py"], cwd=CODE, capture_output=True, text=True)
assert r.returncode == 0, r.stderr[-1500:]
assert os.path.exists(DATA_DIR + "/THPep_train.csv")
print("ok -- layout mirrored, THPep split written")


In [ ]:

# -- Cell 4 -- export the two arms.
#
# export_truncated.py verifies each two ways: exported block j must be
# bit-identical to source block keep[j] (tensor comparison, no forward pass, so
# rotary state cannot confound it), and the loaded model must reproduce the
# original with the same blocks bypassed at runtime.
EXPORT = WORK + "/compressed"
os.makedirs(EXPORT, exist_ok=True)

SELECTIONS = {
    "bisel8": "0,1,2,3,5,6,10,16",     # block 0 + the 7 highest-BI blocks in 1-16
    "trunc8": "0-7",                   # matched-size control
}

ARMS = {}
for name, keep in SELECTIONS.items():
    out = "%s/peptideclm-2-mlm-%s" % (EXPORT, name)
    if not os.path.exists(out + "/model.safetensors"):
        r = subprocess.run(["python", "export_truncated.py", "--out", out,
                            "--keep", keep], cwd=CODE, capture_output=True, text=True)
        print("== %s (keep %s) ==" % (name, keep)); print(r.stdout[-500:])
        if r.returncode != 0:
            print(r.stderr[-1200:]); continue
    ARMS[name] = out

# Matched capacity is the whole point of the comparison; if the files differ in
# size the result is about parameters, not about which blocks were chosen.
sizes = {k: os.path.getsize(v + "/model.safetensors") for k, v in ARMS.items()}
print("\nsizes:", {k: "%.1f MB" % (v / 1e6) for k, v in sizes.items()})
assert len(set(sizes.values())) == 1, "arms differ in size -- not a fair comparison"


In [ ]:

# -- Cell 5 -- run their LoRA classification script on THPep, unmodified.
#
# THPep has no val file, so their script takes the 5-fold CV branch: five training
# runs per job, each predicting the full 122-molecule test set, ensembled by mean
# logit in Cell 6. An 8-block arm took 9.7 min in the earlier run, so both arms
# finish in well under half an hour on two GPUs.
#
# --gpu_index must be passed: their Trainer does devices=[int(args.gpu_index)] on
# the raw argument, whose default is None.
OUT = WORK + "/results/bi_slice"
os.makedirs(OUT, exist_ok=True)
SEED = 101

subprocess.run("rclone copy %s/results/bi_slice %s --transfers 8 -P" % (REMOTE, OUT),
               shell=True, check=False)
todo = [a for a in ARMS if not glob.glob("%s/%s/seed_%d/*_results.csv" % (OUT, a, SEED))]
print("%d arms, %d to run" % (len(ARMS), len(todo)))

running, free, t0 = [], list(range(NGPU)), time.time()
while todo or running:
    while todo and free:
        arm = todo.pop(0); gpu = free.pop(0)
        d = "%s/%s/seed_%d" % (OUT, arm, SEED)
        os.makedirs(d, exist_ok=True)
        cmd = ["python", TRAIN_PY, "--dataset", "THPep", "--gpu", "0",
               "--gpu_index", "0", "--model_name", ARMS[arm],
               "--batch_size", "32", "--seed", str(SEED),
               "--data_dir", DATA_DIR, "--save_path", d,
               "--log_dir", "/tmp/logs/%s" % arm]
        p = subprocess.Popen(cmd, cwd=os.path.dirname(TRAIN_PY),
                             stdout=open(d + "/train.log", "w"),
                             stderr=subprocess.STDOUT,
                             env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)))
        running.append((arm, gpu, p, d))
        print("[%5.1f min] launch %-8s gpu%d" % ((time.time()-t0)/60, arm, gpu))
    time.sleep(20)
    for job in list(running):
        arm, gpu, p, d = job
        if p.poll() is None:
            continue
        running.remove(job); free.append(gpu)
        ok = p.returncode == 0 and glob.glob(d + "/*_results.csv")
        print("[%5.1f min] %-8s -> %s" % ((time.time()-t0)/60, arm,
                                          "ok" if ok else "FAILED rc=%s" % p.returncode))
        if not ok:
            print("".join(open(d + "/train.log").readlines()[-15:]))
print("")
print("done in %.1f min" % ((time.time() - t0) / 60))


In [ ]:

# -- Cell 6 -- read it, with a bootstrap CI.
#
# THPep's test set is 122 molecules, so the CI is roughly +-0.15 and only a large
# effect is readable. Report the interval, not just the point estimate -- the
# earlier slice probe only became interpretable once the CIs were on the table.
from sklearn.metrics import matthews_corrcoef, roc_auc_score, accuracy_score

rng = np.random.default_rng(0)
rows = []
for f in sorted(glob.glob(OUT + "/*/seed_*/*_results.csv")):
    arm = os.path.basename(os.path.dirname(os.path.dirname(f)))
    d = pd.read_csv(f)
    if "fold" in d.columns and d.fold.nunique() > 1:
        d["i"] = d.groupby("fold").cumcount(); g = d.groupby("i")
        y, p = g.true_label.first().values, g.predicted_label.mean().values
    else:
        y, p = d.true_label.values, d.predicted_label.values
    m = matthews_corrcoef(y, (p > 0).astype(int))
    bs = []
    for _ in range(2000):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) < 2:
            continue
        bs.append(matthews_corrcoef(y[i], (p[i] > 0).astype(int)))
    rows.append(dict(arm=arm, keep=SELECTIONS.get(arm, "?"), n=len(y),
                     mcc=round(m, 4),
                     lo=round(float(np.percentile(bs, 2.5)), 4),
                     hi=round(float(np.percentile(bs, 97.5)), 4),
                     auc=round(roc_auc_score(y, p), 4),
                     acc=round(accuracy_score(y, (p > 0).astype(int)), 4)))
res = pd.DataFrame(rows)
print(res.to_string(index=False))

print("\nreference points on THPep, same 5-fold protocol:")
for k, v in [("prefix16 (0-15)", 0.8531), ("warmstart32M", 0.8431),
             ("trunc24 (0-23)", 0.8218), ("trunc8 (0-7), earlier run", 0.8037),
             ("full337M (0-31)", 0.7764), ("trunc31 (0-30)", 0.7642),
             ("their published mlm-large", 0.7557),
             ("mid16 (8-23)", 0.6379), ("bag-of-tokens control", 0.6854),
             ("suffix16 (16-31)", 0.5593)]:
    print("   %-28s %.4f" % (k, v))

if len(res) == 2:
    a, b = res.iloc[0], res.iloc[1]
    print("\n%s - %s = %+.4f" % (a["arm"], b["arm"], a["mcc"] - b["mcc"]))
    sep = a["lo"] > b["hi"] or b["lo"] > a["hi"]
    print("CIs %s -- %s" % ("do NOT overlap" if sep else "overlap",
          "the difference is readable" if sep else
          "indistinguishable at this sample size; a win needs AmpHGT to confirm"))
print("\ntrunc8 here vs 0.8037 from the earlier run is also a reproducibility check:")
print("a large gap would mean THPep run-to-run variance exceeds the CI, and that")
print("every THPep comparison in this project needs re-reading.")

res.to_csv(OUT + "/bi_slice_metrics.csv", index=False)
subprocess.run("rclone copy %s %s/results/bi_slice --drive-chunk-size 64M -P"
               % (OUT, REMOTE), shell=True, check=True)
print("\nuploaded -> %s/results/bi_slice" % REMOTE)
